# Assignment Beta - Question 4 (Student 23K-0003)
LLM sentiment evaluation: accuracy, per-class metrics, Stage 1 vs Stage 2, prompt stability, BLEU/ROUGE/METEOR, perplexity. Pure Python.

In [1]:

import math, statistics as st
labels=['Positive','Negative','Neutral']
S1=[[100,5,15],[8,80,12],[20,10,50]]
S2=[[106,4,10],[6,84,10],[9,7,64]]
def metrics(M):
    n=sum(map(sum,M)); k=len(M); tp=[M[i][i] for i in range(k)]
    rows=[sum(M[i]) for i in range(k)]; cols=[sum(M[i][j] for i in range(k)) for j in range(k)]
    P=[tp[i]/cols[i] for i in range(k)]; R=[tp[i]/rows[i] for i in range(k)]
    F=[2*P[i]*R[i]/(P[i]+R[i]) for i in range(k)]
    acc=sum(tp)/n
    z=1.96; se=math.sqrt(acc*(1-acc)/n); wald=(acc-z*se,acc+z*se)
    w=(acc+z*z/(2*n))/(1+z*z/n); hw=z*math.sqrt(acc*(1-acc)/n+z*z/(4*n*n))/(1+z*z/n); wil=(w-hw,w+hw)
    macro=sum(F)/k; weighted=sum(F[i]*rows[i] for i in range(k))/n
    micro_p=sum(tp)/sum(cols); micro_r=sum(tp)/sum(rows); micro=2*micro_p*micro_r/(micro_p+micro_r)
    return dict(n=n,tp=tp,rows=rows,cols=cols,P=P,R=R,F=F,acc=acc,se=se,wald=wald,wilson=wil,macro=macro,weighted=weighted,micro=micro)
def show(name,M):
    m=metrics(M); print(name)
    print(f"accuracy = {sum(m['tp'])}/{m['n']} = {m['acc']:.4f}; SE = {m['se']:.4f}; Wald 95% CI = ({m['wald'][0]:.4f}, {m['wald'][1]:.4f}); Wilson 95% CI = ({m['wilson'][0]:.4f}, {m['wilson'][1]:.4f})")
    for i,l in enumerate(labels): print(f"{l:9s} P={m['tp'][i]}/{m['cols'][i]}={m['P'][i]:.4f}  R={m['tp'][i]}/{m['rows'][i]}={m['R'][i]:.4f}  F1={m['F'][i]:.4f}")
    print(f"macro-F1={m['macro']:.4f}  weighted-F1={m['weighted']:.4f}  micro-F1={m['micro']:.4f} (micro-F1 = accuracy for single-label multiclass)")
    return m
m1=show('STAGE 1 (zero-shot)',S1); print(); m2=show('STAGE 2 (improved prompt)',S2)


STAGE 1 (zero-shot)
accuracy = 230/300 = 0.7667; SE = 0.0244; Wald 95% CI = (0.7188, 0.8145); Wilson 95% CI = (0.7156, 0.8110)
Positive  P=100/128=0.7812  R=100/120=0.8333  F1=0.8065
Negative  P=80/95=0.8421  R=80/100=0.8000  F1=0.8205
Neutral   P=50/77=0.6494  R=50/80=0.6250  F1=0.6369
macro-F1=0.7546  weighted-F1=0.7659  micro-F1=0.7667 (micro-F1 = accuracy for single-label multiclass)

STAGE 2 (improved prompt)
accuracy = 254/300 = 0.8467; SE = 0.0208; Wald 95% CI = (0.8059, 0.8874); Wilson 95% CI = (0.8015, 0.8830)
Positive  P=106/121=0.8760  R=106/120=0.8833  F1=0.8797
Negative  P=84/95=0.8842  R=84/100=0.8400  F1=0.8615
Neutral   P=64/84=0.7619  R=64/80=0.8000  F1=0.7805
macro-F1=0.8406  weighted-F1=0.8472  micro-F1=0.8467 (micro-F1 = accuracy for single-label multiclass)


In [2]:

print('Stage1 errors:',300-sum(m1['tp']),' Stage2 errors:',300-sum(m2['tp']))
print('Neutral recall: %.3f -> %.3f ; Neutral precision: %.3f -> %.3f'%(m1['R'][2],m2['R'][2],m1['P'][2],m2['P'][2]))
print('Accuracy gain: %.4f (%.1f points); macro-F1 gain: %.4f'%(m2['acc']-m1['acc'],100*(m2['acc']-m1['acc']),m2['macro']-m1['macro']))
p1,p2=m1['acc'],m2['acc']; pp=(p1+p2)/2; zst=(p2-p1)/math.sqrt(pp*(1-pp)*2/300); print('two-proportion z (unpaired, approximate) = %.2f'%zst)


Stage1 errors: 70  Stage2 errors: 46
Neutral recall: 0.625 -> 0.800 ; Neutral precision: 0.649 -> 0.762
Accuracy gain: 0.0800 (8.0 points); macro-F1 gain: 0.0859
two-proportion z (unpaired, approximate) = 2.48


In [3]:

runs=[0.847,0.840,0.853,0.837,0.850]
mean=st.mean(runs); sd=st.stdev(runs); sdp=st.pstdev(runs)
print(f"mean = {mean:.4f}; sample SD (n-1) = {sd:.4f}; population SD = {sdp:.4f}; range = {max(runs)-min(runs):.3f}")
print(f"mean +- SD = {mean:.3f} +- {sd:.4f}; coefficient of variation = {100*sd/mean:.2f}%")
print(f"Deviations from mean: {[round(r-mean,4) for r in runs]}; sum of squares = {sum((r-mean)**2 for r in runs):.7f}")


mean = 0.8454; sample SD (n-1) = 0.0067; population SD = 0.0060; range = 0.016
mean +- SD = 0.845 +- 0.0067; coefficient of variation = 0.80%
Deviations from mean: [0.0016, -0.0054, 0.0076, -0.0084, 0.0046]; sum of squares = 0.0001812


In [4]:

from collections import Counter
ref="the delivery was late but the product quality is good".split()
cand="the product quality is good but delivery was slow".split()
def ngr(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1))
pn=[]; pn_txt=[]
for n in range(1,5):
    c,r=ngr(cand,n),ngr(ref,n); clip=sum(min(v,r[k]) for k,v in c.items()); tot=sum(c.values()); pn.append(clip/tot); pn_txt.append(f"p{n}={clip}/{tot}")
    print(f"BLEU p{n} = {clip}/{tot} = {clip/tot:.4f}")
bp=1 if len(cand)>len(ref) else math.exp(1-len(ref)/len(cand))
bleu=bp*math.exp(sum(math.log(x) for x in pn)/4)
print(f"c={len(cand)}, r={len(ref)}, BP = exp(1-r/c) = {bp:.4f}; BLEU-4 = {bleu:.4f}")
res={}
def rouge_n(n):
    c,r=ngr(cand,n),ngr(ref,n); ov=sum(min(v,r[k]) for k,v in c.items())
    P=ov/sum(c.values()); R=ov/sum(r.values()); F=2*P*R/(P+R) if ov else 0; return ov,P,R,F
for n in (1,2):
    ov,P,R,F=rouge_n(n); res[n]=(ov,P,R,F); print(f"ROUGE-{n}: overlap={ov} P={P:.4f} R={R:.4f} F1={F:.4f}")
def lcs(a,b):
    T=[[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1): T[i][j]=T[i-1][j-1]+1 if a[i-1]==b[j-1] else max(T[i-1][j],T[i][j-1])
    return T[-1][-1]
L=lcs(cand,ref); PL=L/len(cand); RL=L/len(ref); FL=2*PL*RL/(PL+RL)
print(f"ROUGE-L: LCS={L} P={PL:.4f} R={RL:.4f} F1={FL:.4f}")
# METEOR (Banerjee & Lavie 2005): exact matching, alpha=.9 -> Fmean=10PR/(R+9P), penalty=0.5*(chunks/m)^3
# alignment chosen to minimise chunks (cand idx -> ref idx); 'slow' unmatched (no synonym resource used)
align={0:5,1:6,2:7,3:8,4:9,5:4,6:1,7:2}
m=len(align); Pm=m/len(cand); Rm=m/len(ref); Fm=10*Pm*Rm/(Rm+9*Pm)
idx=sorted(align); chunks=1+sum(1 for a,b in zip(idx,idx[1:]) if align[b]!=align[a]+1)
pen=0.5*(chunks/m)**3; meteor=Fm*(1-pen)
print(f"METEOR: m={m}, P={Pm:.4f}, R={Rm:.4f}, Fmean={Fm:.4f}, chunks={chunks}, penalty={pen:.4f}, METEOR={meteor:.4f}")


BLEU p1 = 8/9 = 0.8889
BLEU p2 = 5/8 = 0.6250
BLEU p3 = 3/7 = 0.4286
BLEU p4 = 2/6 = 0.3333
c=9, r=10, BP = exp(1-r/c) = 0.8948; BLEU-4 = 0.4750
ROUGE-1: overlap=8 P=0.8889 R=0.8000 F1=0.8421
ROUGE-2: overlap=5 P=0.6250 R=0.5556 F1=0.5882
ROUGE-L: LCS=5 P=0.5556 R=0.5000 F1=0.5263
METEOR: m=8, P=0.8889, R=0.8000, Fmean=0.8081, chunks=3, penalty=0.0264, METEOR=0.7868


In [5]:

probs=[0.60,0.40,0.50,0.80,0.70,0.30,0.25,0.50,0.20]
lp=[math.log(p) for p in probs]; avg=-sum(lp)/len(lp); ppl=math.exp(avg)
for t,p,l in zip(cand,probs,lp): print(f"{t:9s} p={p:.2f} ln p={l:+.4f}")
print(f"sum ln p = {sum(lp):.4f}; avg NLL = {avg:.4f}; perplexity = exp(avg NLL) = {ppl:.3f}")


the       p=0.60 ln p=-0.5108
product   p=0.40 ln p=-0.9163
quality   p=0.50 ln p=-0.6931
is        p=0.80 ln p=-0.2231
good      p=0.70 ln p=-0.3567
but       p=0.30 ln p=-1.2040
delivery  p=0.25 ln p=-1.3863
was       p=0.50 ln p=-0.6931
slow      p=0.20 ln p=-1.6094
sum ln p = -7.5929; avg NLL = 0.8437; perplexity = exp(avg NLL) = 2.325
